In [0]:
customers_path = "abfss://lakehouse@stretaildeveas001.dfs.core.windows.net/landing/customers/2026/09/06/customers_2026-09-06.csv"

df_customers = (
    spark.read
    .option("header", True)
    .csv(customers_path)
)

display(df_customers)

In [0]:
df_customers.printSchema()

In [0]:
df_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "country"
).show()

In [0]:
df_customers.filter(
    df_customers.country == "PH"
).show()

In [0]:
df_customers.count()

In [0]:
display(
    df_customers.describe()
)

In [0]:
df_customers.columns

In [0]:
df_customers.groupBy("customer_id").count().orderBy("count", ascending=False).show()

In [0]:
from pyspark.sql import functions as F

In [0]:
df_customers_typed = (
    df_customers
    .withColumn(
        "created_at",
        F.to_timestamp("created_at")
    )
)

In [0]:
df_customers_typed.printSchema()


In [0]:
df_customers_bronze = (
    df_customers_typed
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")  # Use this instead of F.input_file_name()
    )
)


In [0]:
display(df_customers_bronze)

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS dbw_retail_lakehouse_dev_eas_001.bronze;

CREATE SCHEMA IF NOT EXISTS dbw_retail_lakehouse_dev_eas_001.silver;

CREATE SCHEMA IF NOT EXISTS dbw_retail_lakehouse_dev_eas_001.gold;

In [0]:
%sql

SHOW SCHEMAS IN dbw_retail_lakehouse_dev_eas_001;

In [0]:
(
    df_customers_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "dbw_retail_lakehouse_dev_eas_001.bronze.customers"
    )
)

In [0]:
%sql

SELECT *
FROM dbw_retail_lakehouse_dev_eas_001.bronze.customers;

In [0]:
%sql

DESCRIBE TABLE
dbw_retail_lakehouse_dev_eas_001.bronze.customers;

In [0]:
%sql

DESCRIBE TABLE
dbw_retail_lakehouse_dev_eas_001.bronze.customers;

In [0]:
%sql
SELECT COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.bronze.customers;